In [7]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#for topic modeling
from gensim import corpora
from gensim.models import LdaModel

#load data
documents = [
 "Rafael Nadal Joins Roger Federer in Missing U.S. Open",
 "Rafael Nadal Is Out of the Australian Open",
 "Biden Announces Virus Measures",
 "Biden's Virus Plans Meet Reality",
 "Where Biden's Virus Plan Stands"
]

In [8]:
stop_words = set(stopwords.words('english')) #create a set of english stopwords
lemmatizer = WordNetLemmatizer() #Initalize a WordNet Lemmatizer

def preprocess_text(text):
    tokens = word_tokenize(text.lower()) #tokenize the text into words and convert into lowercase
    tokens = [token for token in tokens if token.isalnum()] # filter out non-alphaneumeric tokens
    tokens = [token for token in tokens if token not in stop_words] #remove stopwords from the tokens
    tokens = [lemmatizer.lemmatize(token) for token in tokens] #lemmatize each token
    return tokens #return the preprocessed tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents] #preprocess each document in the list
preprocessed_documents

[['rafael', 'nadal', 'join', 'roger', 'federer', 'missing', 'open'],
 ['rafael', 'nadal', 'australian', 'open'],
 ['biden', 'announces', 'virus', 'measure'],
 ['biden', 'virus', 'plan', 'meet', 'reality'],
 ['biden', 'virus', 'plan', 'stand']]

In [9]:
#create a document-term matrix
dictionary = corpora.Dictionary(preprocessed_documents)
#convert each preprocessed document into a bag of words representation using a dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]


In [10]:
#corpus: bag of words representation of the documents
#num_topics: number of topics to be extracted by the model
#id2word=dictionary: dictionary mapping from word IDs to words
#passes: number of passes through the corpus during training
#train on LDA model on the corpus with 4 topics using Gensim's LdaModel class
lda_model = LdaModel(corpus, num_topics=2, id2word=dictionary, passes=15)


In [11]:
#empty lists to store dominant topic labels for each document
article_labels = []

#iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
    #for each document, convert to box representation
    bow = dictionary.doc2bow(doc)
    #get list of topic probabilities
    topics = lda_model.get_document_topics(bow)
    #determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    #appenf to the list
    article_labels.append(dominant_topic)

In [12]:
#create dataframe
df = pd.DataFrame({"Article": documents, "Topic": article_labels})

#print the dataframe
print("Table with articles and topic")
print(df)
print()

Table with articles and topic
                                             Article  Topic
0  Rafael Nadal Joins Roger Federer in Missing U....      0
1         Rafael Nadal Is Out of the Australian Open      0
2                     Biden Announces Virus Measures      1
3                   Biden's Virus Plans Meet Reality      1
4                    Where Biden's Virus Plan Stands      1



In [16]:
#print the top terms for each topic
print("Top terms for each topic: ")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top terms for each topic: 
Topic 0:
- "nadal" (weight: 0.131)
- "open" (weight: 0.131)
- "rafael" (weight: 0.131)
- "missing" (weight: 0.079)
- "roger" (weight: 0.079)
- "join" (weight: 0.079)
- "federer" (weight: 0.079)
- "australian" (weight: 0.079)
- "biden" (weight: 0.027)
- "virus" (weight: 0.027)

Topic 1:
- "virus" (weight: 0.166)
- "biden" (weight: 0.166)
- "plan" (weight: 0.119)
- "reality" (weight: 0.071)
- "meet" (weight: 0.071)
- "measure" (weight: 0.071)
- "announces" (weight: 0.071)
- "stand" (weight: 0.071)
- "rafael" (weight: 0.024)
- "open" (weight: 0.024)

